### Linear Regression
Car price prediction project

The dataset: data.csv from Kaggle

The project plan
We will do the project in several steps:

Get the data and do exploratory data analysis: just look at the data and try to learn more about it.
Prepare the dataset and train a linear regression model on it to predict the price of a car.
Go into the details of how linear regression is implemented — we will actually implement it ourselves.
Evaluate the quality of the model with RMSE. RMSE stands for root mean squared error, a metric for evaluating the quality of model predictions.
Do some feature engineering: the process of creating new features, new characteristics that we can use for our model.
Deal with numerical stability problems and see how to solve them with regularization.
Finally, use the model.

### Step 1: Data preparation

We take the car price dataset and prepare it for machine learning: we download it, load it with pandas, and make the column names and the string values consistent.

In [28]:
import pandas as pd

In [29]:
# Loading the dataset

df = pd.read_csv('data.csv')

In [30]:
df.sample(5)

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
1305,Suzuki,Aerio,2005,regular unleaded,155.0,4.0,AUTOMATIC,all wheel drive,4.0,NaN,Compact,Wagon,26,21,481,17349
7974,Audi,R8,2014,premium unleaded (required),430.0,8.0,AUTOMATED_MANUAL,all wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,23,14,3105,124000
5191,Pontiac,G6,2009,flex-fuel (unleaded/E85),219.0,6.0,AUTOMATIC,front wheel drive,4.0,Flex Fuel,Midsize,Sedan,26,17,210,24710
10032,Cadillac,STS,2011,regular unleaded,302.0,6.0,AUTOMATIC,rear wheel drive,4.0,"Luxury,Performance",Large,Sedan,27,18,1624,50820
6072,Volkswagen,Jetta SportWagen,2013,regular unleaded,170.0,5.0,MANUAL,front wheel drive,4.0,NaN,Compact,Wagon,33,23,873,20595


Making the column names consistent

Looking at the column names, there is some inconsistency. Sometimes there are uppercase letters (Make, Model), sometimes there are spaces (Engine Fuel Type), sometimes underscores (Driven_Wheels).

To keep life simple, let's clean the names: make everything lowercase and replace spaces with underscores.

In [31]:
# String operations on column names

df.columns.str.lower().str.replace(' ', '_')

Index(['make', 'model', 'year', 'engine_fuel_type', 'engine_hp',
       'engine_cylinders', 'transmission_type', 'driven_wheels',
       'number_of_doors', 'market_category', 'vehicle_size', 'vehicle_style',
       'highway_mpg', 'city_mpg', 'popularity', 'msrp'],
      dtype='str')

In [32]:
# Write the result back to the columns attribute:

df.columns = df.columns.str.lower().str.replace(' ', '_')

The values have the same problem as the columns names: sometimes they are all caps (BMW, MANUAL), sometimes not. Let's normalize the string values the same way.

First we need to find out which columns are strings — we cannot apply string methods to numbers. For that we use the dtypes attribute. It tells us, for every column, what type it is:

In [34]:
# Finding the string columns

df.dtypes

make                     str
model                    str
year                   int64
engine_fuel_type         str
engine_hp            float64
engine_cylinders     float64
transmission_type        str
driven_wheels            str
number_of_doors      float64
market_category          str
vehicle_size             str
vehicle_style            str
highway_mpg            int64
city_mpg               int64
popularity             int64
msrp                   int64
dtype: object

In [40]:
# Selecting the string columns

df.dtypes[df.dtypes == 'object']

Series([], dtype: object)

Course material is outdated on this point. The course was written for pandas 1.x/2.x, where text columns had the object dtype. Pandas 3.0 stores text in a dedicated string type that displays as str.

In [43]:
# # Selecting the string columns

df.dtypes[df.dtypes == 'str']

make                 str
model                str
engine_fuel_type     str
transmission_type    str
driven_wheels        str
market_category      str
vehicle_size         str
vehicle_style        str
dtype: object

In [ ]:
#  Take the index of this series and convert it to a Python list. No particular reason, it just looks nicer:

df.dtypes[df.dtypes == 'str'].index

Index(['make', 'model', 'engine_fuel_type', 'transmission_type',
       'driven_wheels', 'market_category', 'vehicle_size', 'vehicle_style'],
      dtype='str')

In [48]:
# Python list

strings = list(df.dtypes[df.dtypes == 'str'].index)     # Or, df.select_dtypes(include='str').columns.tolist()
strings

['make',
 'model',
 'engine_fuel_type',
 'transmission_type',
 'driven_wheels',
 'market_category',
 'vehicle_size',
 'vehicle_style']

Normalizing the string values

Now that we have all the string column names, we loop over them and apply the same transformation we used for the column names: lowercase everything and replace spaces with underscores, writing the result back to the DataFrame:

In [49]:
# Normalizing the string values

for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

After the loop the data looks cleaner. Everything is lowercase, and spaces in values are replaced with underscores:
The dataset is prepared

In [50]:
df.sample(5)

,make,model,year,engine_fuel_type,engine_hp,engine_cylinders,transmission_type,driven_wheels,number_of_doors,market_category,vehicle_size,vehicle_style,highway_mpg,city_mpg,popularity,msrp
1830,volkswagen,beetle_convertible,2015,regular_unleaded,170.0,4.0,automatic,front_wheel_drive,2.0,NaN,compact,convertible,32,24,873,29595
2562,honda,civic,2015,regular_unleaded,143.0,4.0,manual,front_wheel_drive,2.0,NaN,compact,coupe,36,28,2202,18290
653,bmw,6_series,2015,premium_unleaded_(required),445.0,8.0,automatic,all_wheel_drive,2.0,"luxury,high-performance",midsize,convertible,24,16,3916,97900
10385,hyundai,tiburon,2006,regular_unleaded,138.0,4.0,automatic,front_wheel_drive,2.0,hatchback,compact,2dr_hatchback,27,19,1439,17095
5969,lexus,is_250,2013,premium_unleaded_(required),204.0,6.0,automatic,all_wheel_drive,4.0,luxury,midsize,sedan,27,21,454,37525


### Step 2: 